#### Configure ADLS Connection

In [0]:
storage_account_name = "stsupplychainlak"
container_name = "raw-landing"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    dbutils.secrets.get(
        scope="adls-scope",
        key="storage-account-key"
    )
)

#### Test the Connection

In [0]:
display(
    dbutils.fs.ls(
        "abfss://raw-landing@stsupplychainlak.dfs.core.windows.net/"
    )
)

path,name,size,modificationTime
abfss://raw-landing@stsupplychainlak.dfs.core.windows.net/ config/,config/,0,1781170519000
abfss://raw-landing@stsupplychainlak.dfs.core.windows.net/taxi_zone_lookup/,taxi_zone_lookup/,0,1781179336000
abfss://raw-landing@stsupplychainlak.dfs.core.windows.net/yellow_taxi/,yellow_taxi/,0,1781172625000


#### Read Yellow Taxi Parquet Files

In [0]:
yellow_taxi_df = spark.read.parquet(
    "abfss://raw-landing@stsupplychainlak.dfs.core.windows.net/yellow_taxi/*.parquet"
)

display(yellow_taxi_df)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1,2025-10-01T00:15:32,2025-10-01T01:04:03,1,17.2,2,N,132,107,1,70.0,5.0,0.5,0.0,6.94,1.0,83.44,2.5,1.75,0.75
7,2025-10-01T00:00:08,2025-10-01T00:00:08,1,5.0,1,N,107,225,1,28.2,0.0,0.5,8.49,0.0,1.0,42.44,2.5,0.0,0.75
2,2025-10-01T00:08:54,2025-10-01T00:14:44,1,2.75,1,N,263,229,1,12.8,1.0,0.5,3.71,0.0,1.0,22.26,2.5,0.0,0.75
1,2025-10-01T00:58:48,2025-10-01T01:04:40,1,1.3,1,N,211,231,2,7.9,4.25,0.5,0.0,0.0,1.0,13.65,2.5,0.0,0.75
2,2025-10-01T00:39:51,2025-10-01T00:49:40,1,2.88,1,N,230,151,1,14.2,1.0,0.5,3.99,0.0,1.0,23.94,2.5,0.0,0.75
1,2025-10-01T00:30:54,2025-10-01T00:37:50,1,1.6,1,N,237,142,2,9.3,3.5,0.5,0.0,0.0,1.0,14.3,2.5,0.0,0.0
2,2025-10-01T00:10:12,2025-10-01T00:17:29,1,2.81,1,N,142,166,1,12.8,1.0,0.5,3.56,0.0,1.0,21.36,2.5,0.0,0.0
2,2025-10-01T00:48:19,2025-10-01T01:01:02,1,2.17,1,N,230,246,1,13.5,1.0,0.5,0.0,0.0,1.0,19.25,2.5,0.0,0.75
2,2025-10-01T00:08:44,2025-10-01T00:24:17,1,3.71,1,N,140,7,1,19.1,1.0,0.5,6.21,0.0,1.0,31.06,2.5,0.0,0.75
2,2025-10-01T00:25:23,2025-10-01T00:33:02,1,1.2,1,N,234,249,2,8.6,1.0,0.5,0.0,0.0,1.0,14.35,2.5,0.0,0.75


#### Read Taxi Zone Lookup CSV

In [0]:
zone_lookup_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(
        "abfss://raw-landing@stsupplychainlak.dfs.core.windows.net/taxi_zone_lookup/taxi_zone_lookup.csv"
    )

display(zone_lookup_df)

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


#### Write Yellow Taxi Bronze Delta

In [0]:
yellow_taxi_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://bronze@stsupplychainlak.dfs.core.windows.net/yellow_taxi"
    )

#### Write Taxi Zone Bronze Delta

In [0]:
zone_lookup_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "abfss://bronze@stsupplychainlak.dfs.core.windows.net/taxi_zone_lookup"
    )

#### Validate Bronze Layer

In [0]:
bronze_yellow = spark.read.format("delta").load(
    "abfss://bronze@stsupplychainlak.dfs.core.windows.net/yellow_taxi"
)

display(bronze_yellow)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1,2025-07-01T00:29:37,2025-07-01T00:45:30,1,7.3,1,N,138,74,1,29.6,7.75,0.5,9.0,6.94,1.0,54.79,0.0,1.75,0.0
1,2025-07-01T00:23:28,2025-07-01T01:07:44,1,17.7,2,N,132,142,1,70.0,4.25,0.5,5.0,0.0,1.0,80.75,2.5,1.75,0.0
2,2025-07-01T00:53:50,2025-07-01T01:27:12,1,9.98,1,N,138,48,1,43.6,6.0,0.5,10.87,0.0,1.0,66.97,2.5,1.75,0.75
2,2025-07-01T00:58:49,2025-07-01T01:15:55,1,10.27,1,N,138,229,1,38.7,6.0,0.5,14.1,6.94,1.0,72.24,2.5,1.75,0.75
2,2025-07-01T00:09:22,2025-07-01T00:23:54,1,2.94,1,N,211,97,1,17.0,1.0,0.5,3.0,0.0,1.0,25.75,2.5,0.0,0.75
1,2025-07-01T00:39:14,2025-07-01T00:55:21,1,11.8,1,N,132,155,1,44.3,1.0,0.5,14.05,0.0,1.0,60.85,0.0,0.0,0.0
2,2025-07-01T00:15:26,2025-07-01T00:29:39,1,3.87,1,N,79,263,1,17.7,1.0,0.5,4.69,0.0,1.0,28.14,2.5,0.0,0.75
2,2025-07-01T00:40:58,2025-07-01T00:44:15,1,0.85,1,N,140,262,1,5.8,1.0,0.5,2.16,0.0,1.0,12.96,2.5,0.0,0.0
2,2025-07-01T00:28:12,2025-07-01T00:39:49,2,2.54,1,N,114,50,1,14.2,1.0,0.5,3.99,0.0,1.0,23.94,2.5,0.0,0.75
2,2025-07-01T00:38:17,2025-07-01T00:55:44,1,6.37,1,N,132,197,1,26.8,1.0,0.5,5.86,0.0,1.0,36.91,0.0,1.75,0.0


In [0]:
bronze_zone = spark.read.format("delta").load(
    "abfss://bronze@stsupplychainlak.dfs.core.windows.net/taxi_zone_lookup"
)

display(bronze_zone)

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


In [0]:
bronze_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [0]:
bronze_zone.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [0]:
print(f"Total Rows: {bronze_yellow.count()}")
print(f"Total Columns: {len(bronze_yellow.columns)}")

Total Rows: 39547664
Total Columns: 20


In [0]:
print(f"Total Rows: {bronze_zone.count()}")
print(f"Total Columns: {len(bronze_zone.columns)}")

Total Rows: 265
Total Columns: 4
